In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
import seaborn as sns
import matplotlib.pyplot as plt


In [17]:

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 300)


In [18]:
# load raw data

df_raw = pd.read_csv('food_prediction_raw.csv')
df_raw.head()

,date,category_name,item_id,sold_quantity,price,revenue,store_id,month,day_of_week,week,zipcode,temperature,wind_speed,wind_degree,wind_dir,weather_description,precip,humidity,visibility,pressure,cloudcover,dewpoint,windgust,uv_index,date_zipcode,special_day,school_holiday,std_holiday
0,2025-04-01,Angebot Brötchen,547,15.0,0.0,0.0,0,4,1,14,52062,7.541667,16.5,78.625,E,Clear,0.0,64.458333,10.0,1025.75,5.833333,0.75,25.958333,2.166667,520622025-04-01,1,0,0
1,2025-04-01,Angebot Feinbäckerei,203,28.0,0.0,0.0,0,4,1,14,52062,7.541667,16.5,78.625,E,Clear,0.0,64.458333,10.0,1025.75,5.833333,0.75,25.958333,2.166667,520622025-04-01,1,0,0
2,2025-04-01,Angebot Heißgetränke,346,25.0,0.0,0.0,0,4,1,14,52062,7.541667,16.5,78.625,E,Clear,0.0,64.458333,10.0,1025.75,5.833333,0.75,25.958333,2.166667,520622025-04-01,1,0,0
3,2025-04-01,Angebot Heißgetränke,445,5.0,1.4,7.0,0,4,1,14,52062,7.541667,16.5,78.625,E,Clear,0.0,64.458333,10.0,1025.75,5.833333,0.75,25.958333,2.166667,520622025-04-01,1,0,0
4,2025-04-01,Angebot Snack,330,5.0,0.0,0.0,0,4,1,14,52062,7.541667,16.5,78.625,E,Clear,0.0,64.458333,10.0,1025.75,5.833333,0.75,25.958333,2.166667,520622025-04-01,1,0,0


## Creating Lag Features

__Issue__: No continuous time range__
+ If an item is not sold on a certain date/store there is no record and thus no date. That breaks the date range

__Solution:__ we first have to create a consecutive date range, to get correct lag features afterwards.
+ Lags:
    * t-1 
    * t-7
    * ~~t-14~~

In [19]:
# sort new copy in preperatin for correct lag features
df = df_raw.sort_values(['store_id', 'item_id', 'date']).copy()

# convert date
df['date'] = pd.to_datetime(df.date)

# deleting cols month, day_of_week, week  --> because after reindexing we have new rows of these cols with NaN --> we add them later again after reindexing
df = df.drop(columns=['month', 'day_of_week', 'week'])

df.head()

,date,category_name,item_id,sold_quantity,price,revenue,store_id,zipcode,temperature,wind_speed,wind_degree,wind_dir,weather_description,precip,humidity,visibility,pressure,cloudcover,dewpoint,windgust,uv_index,date_zipcode,special_day,school_holiday,std_holiday
39961,2025-04-05,Kuchen,1,0.0,12.00,0.00,0,52062,11.916667,14.333333,51.833333,NE,Sunny,0.000000,59.875000,10.000000,1017.666667,13.583333,3.833333,21.916667,2.625000,520622025-04-05,0,0,0
104462,2025-04-12,Kuchen,1,0.0,12.00,0.00,0,52062,14.208333,13.833333,139.875000,SE,Sunny,0.004167,60.583333,10.000000,1012.000000,12.041667,6.291667,21.583333,2.916667,520622025-04-12,0,1,0
267089,2025-05-03,Kuchen,1,0.0,12.00,0.00,0,52062,13.416667,8.583333,264.500000,WNW,Light rain shower,0.270833,91.333333,8.291667,1011.583333,68.583333,12.041667,12.833333,2.250000,520622025-05-03,0,0,0
396022,2025-05-17,Kuchen,1,1.0,11.25,11.25,0,52062,11.541667,8.500000,304.375000,N,Sunny,0.000000,71.291667,8.000000,1019.375000,24.750000,5.916667,12.750000,2.666667,520622025-05-17,0,0,0
460492,2025-05-24,Kuchen,1,1.0,11.25,11.25,0,52062,10.125000,19.666667,214.250000,SSW,Partly cloudy,0.166667,74.833333,8.916667,1016.458333,78.125000,5.541667,30.750000,2.000000,520622025-05-24,0,0,0


In [ ]:
df.info()

In [20]:
df = df.set_index(['store_id', 'item_id', 'date'])
df.head(15)

category_name  sold_quantity  price  revenue  \
store_id item_id date                                                      
0        1       2025-04-05        Kuchen            0.0  12.00     0.00   
                 2025-04-12        Kuchen            0.0  12.00     0.00   
                 2025-05-03        Kuchen            0.0  12.00     0.00   
                 2025-05-17        Kuchen            1.0  11.25    11.25   
                 2025-05-24        Kuchen            1.0  11.25    11.25   
                 2025-06-07        Kuchen            0.0  12.00     0.00   
                 2025-06-28        Kuchen            0.0  12.00     0.00   
         7       2025-04-01      Brötchen            9.0   0.70     6.30   
                 2025-04-02      Brötchen            3.0   0.70     2.10   
                 2025-04-03      Brötchen           13.0   0.70     9.10   
                 2025-04-04      Brötchen            2.0   0.70     1.40   
                 2025-04-05      Brötchen           18.0   0.70    12.60   
                 2025-04-07      Brötchen            7.0   0.70     4.90   
                 2025-04-08      Brötchen            2.0   0.70     1.40   
                 2025-04-11      Brötchen           12.0   0.70     8.40   

                             zipcode  temperature  wind_speed  wind_degree  \
store_id item_id date                                                        
0        1       2025-04-05    52062    11.916667   14.333333    51.833333   
                 2025-04-12    52062    14.208333   13.833333   139.875000   
                 2025-05-03    52062    13.416667    8.583333   264.500000   
                 2025-05-17    52062    11.541667    8.500000   304.375000   
                 2025-05-24    52062    10.125000   19.666667   214.250000   
                 2025-06-07    52062    13.500000   20.291667   225.166667   
                 2025-06-28    52062    22.541667   16.125000   259.125000   
         7       2025-04-01    52062     7.541667   16.500000    78.625000   
                 2025-04-02    52062     9.791667   22.083333    82.333333   
                 2025-04-03    52062    12.208333   17.291667   104.500000   
                 2025-04-04    52062    12.833333   11.291667    85.458333   
                 2025-04-05    52062    11.916667   14.333333    51.833333   
                 2025-04-07    52062     6.458333   10.166667    74.125000   
                 2025-04-08    52062     8.458333    8.750000    71.083333   
                 2025-04-11    52062    10.375000    6.500000   221.833333   

                            wind_dir   weather_description    precip  \
store_id item_id date                                                  
0        1       2025-04-05       NE                 Sunny  0.000000   
                 2025-04-12       SE                 Sunny  0.004167   
                 2025-05-03      WNW     Light rain shower  0.270833   
                 2025-05-17        N                 Sunny  0.000000   
                 2025-05-24      SSW         Partly cloudy  0.166667   
                 2025-06-07       SW  Patchy rain possible  0.208333   
                 2025-06-28        W                 Clear  0.000000   
         7       2025-04-01        E                 Clear  0.000000   
                 2025-04-02        E                 Clear  0.000000   
                 2025-04-03      ESE                 Sunny  0.000000   
                 2025-04-04      ESE                 Clear  0.000000   
                 2025-04-05       NE                 Sunny  0.000000   
                 2025-04-07      ESE                 Clear  0.000000   
                 2025-04-08      ESE                 Sunny  0.000000   
                 2025-04-11       NW                 Clear  0.000000   

                              humidity  visibility     pressure  cloudcover  \
store_id item_id date                                                         
0        1       2025-04-

In [21]:
# creating full date range for lags

all_dates = pd.date_range(df.index.get_level_values('date').min(), df.index.get_level_values('date').max(), freq='D')
all_stores = df.index.get_level_values('store_id').unique()
all_items = df.index.get_level_values('item_id').unique()

new_index = pd.MultiIndex.from_product([all_stores, all_items, all_dates],  names=['store_id', 'item_id', 'date'])

df = df.reindex(new_index)

# filling NaN in target with 0 for training target must not be NaN
df['sold_quantity'] = df['sold_quantity'].fillna(0)


In [22]:
# check
df.head(10)
#df.query('item_id == 7').head(20).info()

category_name  sold_quantity  price  revenue  \
store_id item_id date                                                      
0        1       2025-04-01           NaN            0.0    NaN      NaN   
                 2025-04-02           NaN            0.0    NaN      NaN   
                 2025-04-03           NaN            0.0    NaN      NaN   
                 2025-04-04           NaN            0.0    NaN      NaN   
                 2025-04-05        Kuchen            0.0   12.0      0.0   
                 2025-04-06           NaN            0.0    NaN      NaN   
                 2025-04-07           NaN            0.0    NaN      NaN   
                 2025-04-08           NaN            0.0    NaN      NaN   
                 2025-04-09           NaN            0.0    NaN      NaN   
                 2025-04-10           NaN            0.0    NaN      NaN   

                             zipcode  temperature  wind_speed  wind_degree  \
store_id item_id date                                                        
0        1       2025-04-01      NaN          NaN         NaN          NaN   
                 2025-04-02      NaN          NaN         NaN          NaN   
                 2025-04-03      NaN          NaN         NaN          NaN   
                 2025-04-04      NaN          NaN         NaN          NaN   
                 2025-04-05  52062.0    11.916667   14.333333    51.833333   
                 2025-04-06      NaN          NaN         NaN          NaN   
                 2025-04-07      NaN          NaN         NaN          NaN   
                 2025-04-08      NaN          NaN         NaN          NaN   
                 2025-04-09      NaN          NaN         NaN          NaN   
                 2025-04-10      NaN          NaN         NaN          NaN   

                            wind_dir weather_description  precip  humidity  \
store_id item_id date                                                        
0        1       2025-04-01      NaN                 NaN     NaN       NaN   
                 2025-04-02      NaN                 NaN     NaN       NaN   
                 2025-04-03      NaN                 NaN     NaN       NaN   
                 2025-04-04      NaN                 NaN     NaN       NaN   
                 2025-04-05       NE               Sunny     0.0    59.875   
                 2025-04-06      NaN                 NaN     NaN       NaN   
                 2025-04-07      NaN                 NaN     NaN       NaN   
                 2025-04-08      NaN                 NaN     NaN       NaN   
                 2025-04-09      NaN                 NaN     NaN       NaN   
                 2025-04-10      NaN                 NaN     NaN       NaN   

                             visibility     pressure  cloudcover  dewpoint  \
store_id item_id date                                                        
0        1       2025-04-01         NaN          NaN         NaN       NaN   
                 2025-04-02         NaN          NaN         NaN       NaN   
                 2025-04-03         NaN          NaN         NaN       NaN   
                 2025-04-04         NaN          NaN         NaN       NaN   
                 2025-04-05        10.0  1017.666667   13.583333  3.833333   
                 2025-04-06         NaN          NaN         NaN       NaN   
                 2025-04-07         NaN          NaN         NaN       NaN   
                 2025-04-08         NaN          NaN         NaN       NaN   
                 2025-04-09         NaN          NaN         NaN       NaN   
                 2025-04-10         NaN          NaN         NaN       NaN   

                              windgust  uv_index     date_zipcode  \
store_id item_id date                                               
0        1       2025-04-01        NaN       NaN              NaN   
                 2025-04-02        NaN       NaN              NaN   
                 2025-04-03 

In [23]:
# creating lags

lags = [1, 3, 7]

for lag in lags:
    df[f'lag_{lag}'] = df.groupby(['store_id', 'item_id'])['sold_quantity'].shift({lag})


In [24]:
# missing values in lag 1, 3, 7

# proportion of data loss after deleting rows where lags are missing
print('Data loss after removing NaN in lags:\n')
print(df.isna().sum() / df.shape[0])

# removing lags with missing values
lag_cols = ['lag_1', 'lag_3', 'lag_7']
df = df.dropna(subset=lag_cols)
print('-'*40)
print('--- removing missing values in lags... ---- ')
print('-'*40)
print('\nMissing values in lags after removing:\n')
print(df.isna().sum() / df.shape[0])



Data loss after removing NaN in lags:

category_name          0.845533
sold_quantity          0.000000
price                  0.845533
revenue                0.845533
zipcode                0.845533
temperature            0.845533
wind_speed             0.845533
wind_degree            0.845533
wind_dir               0.845533
weather_description    0.845533
precip                 0.845533
humidity               0.845533
visibility             0.845533
pressure               0.845533
cloudcover             0.845533
dewpoint               0.845533
windgust               0.845533
uv_index               0.845533
date_zipcode           0.845533
special_day            0.845533
school_holiday         0.845533
std_holiday            0.845533
lag_1                  0.010989
lag_3                  0.032967
lag_7                  0.076923
dtype: float64
----------------------------------------
--- removing missing values in lags... ---- 
----------------------------------------

Missing values in 

In [ ]:
# check
df.iloc[2000000:2000100]

## Creating Rolling Feature

__rollings__
* mean_7
* ~~mean_14~~
* median_7
+ difference between mean / median (when high value than more outliers)
+ ~~7-day trend~~
* sum_7
* std_7

In [25]:
# creating rollings

# reset for better handling
df = df.reset_index()

# less writing
shifted_group = df.groupby(['store_id', 'item_id'])['sold_quantity'].shift(1)

#...mean and median
df['rolling_mean_7'] = shifted_group.rolling(7).mean()
df['rolling_median_7'] = shifted_group.rolling(7).median()

# rolling mean 14 days back
# df['rolling_mean_14'] = shifted_group.rolling(14).mean()

# ...sum
df['rolling_sum_7'] = shifted_group.rolling(7).sum()

# ...std
df['rolling_std_7'] = shifted_group.rolling(7).std()

# last 7 day trend
# df['trend_7'] = df.rolling_mean_7 - df.rolling_mean_14 / (df['rolling_mean_14'] + 1e-8)

# mean median diff -> low == stable sold_qty, high == outliers
df['mean_median_diff_7'] = df['rolling_mean_7'] - df['rolling_median_7']

# reset multiindex
df = df.set_index(['store_id', 'item_id', 'date'])

In [26]:
# check
df.tail()



category_name  sold_quantity  price  revenue  \
store_id item_id date                                                      
59       849     2025-06-26           NaN            0.0    NaN      NaN   
                 2025-06-27           NaN            0.0    NaN      NaN   
                 2025-06-28           NaN            0.0    NaN      NaN   
                 2025-06-29           NaN            0.0    NaN      NaN   
                 2025-06-30           NaN            0.0    NaN      NaN   

                             zipcode  temperature  wind_speed  wind_degree  \
store_id item_id date                                                        
59       849     2025-06-26      NaN          NaN         NaN          NaN   
                 2025-06-27      NaN          NaN         NaN          NaN   
                 2025-06-28      NaN          NaN         NaN          NaN   
                 2025-06-29      NaN          NaN         NaN          NaN   
                 2025-06-30      NaN          NaN         NaN          NaN   

                            wind_dir weather_description  precip  humidity  \
store_id item_id date                                                        
59       849     2025-06-26      NaN                 NaN     NaN       NaN   
                 2025-06-27      NaN                 NaN     NaN       NaN   
                 2025-06-28      NaN                 NaN     NaN       NaN   
                 2025-06-29      NaN                 NaN     NaN       NaN   
                 2025-06-30      NaN                 NaN     NaN       NaN   

                             visibility  pressure  cloudcover  dewpoint  \
store_id item_id date                                                     
59       849     2025-06-26         NaN       NaN         NaN       NaN   
                 2025-06-27         NaN       NaN         NaN       NaN   
                 2025-06-28         NaN       NaN         NaN       NaN   
                 2025-06-29         NaN       NaN         NaN       NaN   
                 2025-06-30         NaN       NaN         NaN       NaN   

                             windgust  uv_index date_zipcode  special_day  \
store_id item_id date                                                       
59       849     2025-06-26       NaN       NaN          NaN          NaN   
                 2025-06-27       NaN       NaN          NaN          NaN   
                 2025-06-28       NaN       NaN          NaN          NaN   
                 2025-06-29       NaN       NaN          NaN          NaN   
                 2025-06-30       NaN       NaN          NaN          NaN   

                             school_holiday  std_holiday  lag_1  lag_3  lag_7  \
store_id item_id date                                                           
59       849     2025-06-26             NaN          NaN    0.0    0.0    0.0   
                 2025-06-27             NaN          NaN    0.0    0.0    0.0   
                 2025-06-28             NaN          NaN    0.0    0.0    0.0   
                 2025-06-29             NaN          NaN    0.0    0.0    0.0   
                 2025-06-30             NaN          NaN    0.0    0.0    0.0   

                             rolling_mean_7  rolling_median_7  rolling_sum_7  \
store_id item_id date                                                          
59       849     2025-06-26             0.0               0.0            0.0   
                 2025-06-27             0.0               0.0            0.0   
                 2025-06-28             0.0               0.0            0.0   
                 2025-06-29             0.0               0.0            0.0   
                 2025-06-30             0.0               0.0            0.0   

                             rolling_std_7  mean_median_diff_7  
store_id item_id date                                           
59       849     2025-06-26       0.000047                 0.0  
           

## (Re-)Creating Temporal Features
from 'date'
* month
* week
* day_of_week


In [27]:
# reset multiindex first - otherwise .isocalender().week will not work
df = df.reset_index()

# new time features
df['month'] = df.date.dt.month
df['week'] = df.date.dt.isocalendar()['week']
df['day_of_week'] = df.date.dt.dayofweek

# setting back to mulitindex
df = df.set_index(['store_id', 'item_id', 'date'])


In [79]:
# check

# needed for training: change non-numerical dtypes to ´category´
cat_list = ['category_name', 'wind_dir', 'weather_description', 'date_zipcode']

for col in cat_list:
    df[col] = df[col].astype('category')


<class 'pandas.DataFrame'>
MultiIndex: 4631760 entries, (np.int64(0), np.int64(1), Timestamp('2025-04-08 00:00:00')) to (np.int64(59), np.int64(849), Timestamp('2025-06-30 00:00:00'))
Data columns (total 33 columns):
 #   Column               Dtype   
---  ------               -----   
 0   category_name        category
 1   sold_quantity        float64 
 2   price                float64 
 3   revenue              float64 
 4   zipcode              float64 
 5   temperature          float64 
 6   wind_speed           float64 
 7   wind_degree          float64 
 8   wind_dir             category
 9   weather_description  category
 10  precip               float64 
 11  humidity             float64 
 12  visibility           float64 
 13  pressure             float64 
 14  cloudcover           float64 
 15  dewpoint             float64 
 16  windgust             float64 
 17  uv_index             float64 
 18  date_zipcode         category
 19  special_day          float64 
 20  school_h

# __weather__

In [ ]:
df_weather = pd.read_parquet('20260317_121949_weather.parquet', 
                             engine='fastparquet', 
                             columns=['date', 'time', 'zipcode', 'temperature', 'wind_speed', 'wind_degree',
       'wind_dir', 'weather_code', 'weather_description', 'precip', 'humidity',
       'visibility', 'pressure', 'cloudcover', 'heatindex', 'dewpoint',
       'windchill', 'windgust', 'feelslike', 'uv_index']
                             )
#print(df_weather.columns)
df_weather_10k = df_weather.sample(10000)

In [ ]:
weather_corr = df_weather_10k.corr(numeric_only=True).abs()
plt.figure(figsize=(12,6))
cols_temp = ['temperature', 'feelslike', 'windchill', 'heatindex']
sns.heatmap(weather_corr, annot=True, vmin=0.5, vmax=0.99, center=0.7)
plt.tight_layout()
plt.show()

cols_to_keep = ['feelslike', 'wind_gust', 'weather_code', 'precip', 'humidity', 'visibility', 'pressure', 'cloudcover', 'uv_index']
cols_to_del = ['temperature',  'windchill', 'heatindex', 'wind_speed', 'dewpoint',]


In [ ]:
# remaining weather cols 
df_weather = df_weather.drop(columns=cols_to_del)

# weather end

## __Create Train/Test Set__

In [80]:
# counting all days in the dataset
all_days = df.index.get_level_values('date').nunique()

# 80 % for train set
days_for_train = int(all_days * 0.8)
days_for_test = all_days - days_for_train #int(all_days * 0.2)

# get date for split
start_date = df.index.get_level_values('date').min()
end_date = df.index.get_level_values('date').max()

split_date = start_date + pd.Timedelta(days = days_for_train)


# manual split

# 80 % train
train = df[df.index.get_level_values('date') < split_date]

# 20 % test
test  = df[df.index.get_level_values('date') >= split_date]

print('Train:', train.shape, train.index.get_level_values('date').min(), train.index.get_level_values('date').max())
print('Test:', test.shape, test.index.get_level_values('date').min(), test.index.get_level_values('date').max())

Train: (3694380, 33) 2025-04-08 00:00:00 2025-06-13 00:00:00
Test: (937380, 33) 2025-06-14 00:00:00 2025-06-30 00:00:00


In [81]:
# create exploratory and target variables

# train
X_train = train.drop('sold_quantity', axis=1)
y_train = train['sold_quantity']

# test
X_test = test.drop('sold_quantity', axis=1)
y_test = test['sold_quantity']

In [66]:
## X_test.values

## __Model 1st try__

In [87]:
import xgboost
import lightgbm
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [91]:
# xgboost 
xgb = xgboost.XGBRegressor(tree_method='hist', enable_categorical=True)
xgb.fit(X_train, y_train)




,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [96]:
y_pred_xgb = xgb.predict(X_test)
print('XGBoost:')
print('MAE:', mean_absolute_error(y_test, y_pred_xgb))
print('RMSE:', np.sqrt(mean_squared_error(y_test, y_pred_xgb)))

XGBoost:
MAE: 0.24497083723464275
RMSE: 7.218232352426399


In [118]:
# lightgbm
lgb = lightgbm.LGBMRegressor(n_estimators=1000, subsample_for_bin=100000, random_state=42, metric='rmse')
lgb.fit(X_train, y_train)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.709687 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5970
[LightGBM] [Info] Number of data points in the train set: 3694380, number of used features: 32
[LightGBM] [Info] Start training from score 2.508079


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,1000
,subsample_for_bin,100000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [121]:
y_pred_lgb = lgb.predict(X_test)
print('LighGBM Test:')
print('MAE:', mean_absolute_error(y_test, y_pred_lgb))
print('RMSE:', np.sqrt(mean_squared_error(y_test, y_pred_lgb)))

y_pred_lgb_train = lgb.predict(X_train)
print('LighGBM Train:')
print('MAE:', mean_absolute_error(y_train, y_pred_lgb_train))
print('RMSE:', np.sqrt(mean_squared_error(y_train, y_pred_lgb_train)))



LighGBM Test:
MAE: 0.13558927561831013
RMSE: 3.3476833298745823
LighGBM Train:
MAE: 0.07264773970708154
RMSE: 0.6159787734437476


In [ ]:
# LightGBM:
MAE: 0.20288126272325116
RMSE: 3.2958418891273156

LighGBM:
MAE: 0.1459244602906927
RMSE: 3.3463232821532256